In [ ]:
#| default_exp predicttarg

In [ ]:
#| export
from rs3 import targetfeat
from rs3.seq import load_booster_model

In [ ]:
import lightgbm
import pandas as pd
from rs3 import targetdata
from scipy import stats
import numpy as np

In [ ]:
import os
import multiprocessing
max_n_jobs = multiprocessing.cpu_count()

In [ ]:
#| export
def load_target_model(lite=False):
    """Load rule set 3 target model"""
    return load_booster_model('target_lite_model' if lite else 'target_model')

In [ ]:
assert len(load_target_model(lite=True).feature_names) == 29
assert len(load_target_model(lite=False).feature_names) == 47
# both target models carry the median vector from the pickled SimpleImputer
assert load_target_model(lite=True).medians is not None
assert load_target_model(lite=False).medians is not None

In [ ]:
#| export
def predict_target(design_df, aa_subseq_df, domain_feature_df=None,
                   conservation_feature_df=None, id_cols=None):
    """Make predictions using the Rule Set 3 target model. Note that if the protein_domain_df
    or conservation_df are not supplied, then the lite model will be used, otherwise the full model is used.

    :param design_df: DataFrame
    :param aa_subseq_df: DataFrame
    :param domain_feature_df: DataFrame
    :param id_cols: list or str
    :return: list
    """
    if (domain_feature_df is None) or (conservation_feature_df is None):
        lite = True
        domain_feature_df = None
        conservation_feature_df = None
    else:
        lite = False
    model = load_target_model(lite=lite)
    if id_cols is None:
        id_cols = ['sgRNA Context Sequence', 'Target Cut Length', 'Target Transcript', 'Orientation']
    target_feature_df, target_feature_cols = targetfeat.merge_feature_dfs(design_df,
                                                                          aa_subseq_df=aa_subseq_df,
                                                                          domain_df=domain_feature_df,
                                                                          conservation_df=conservation_feature_df,
                                                                          id_cols=id_cols)
    missing = [c for c in model.feature_names if c not in target_feature_cols]
    if missing:
        raise ValueError('merge_feature_dfs did not produce the features this model '
                         'was trained on; missing: ' + ', '.join(missing))
    X_target = target_feature_df[target_feature_cols]
    predictions = model.predict(X_target)
    return predictions

In [ ]:
design_df = pd.read_table('test_data/sgrna-designs.txt')
design_targ_df = targetfeat.add_target_columns(design_df)
id_cols = ['sgRNA Context Sequence', 'Target Cut Length', 'Target Transcript', 'Orientation']

In [ ]:
## aa sequences
aa_seq_df = pd.read_parquet('test_data/target_data/aa_seqs.pq', engine='pyarrow')

In [ ]:
#| notest
# The same data, fetched live from Ensembl.
aa_seq_df = targetdata.build_transcript_aa_seq_df(design_df, n_jobs=2)

In [ ]:
aa_subseq_df = targetfeat.get_aa_subseq_df(sg_designs=design_targ_df, aa_seq_df=aa_seq_df, width=16,
                                           id_cols=id_cols)
aa_subseq_df

In [ ]:
## domains
domain_df = pd.read_parquet('test_data/target_data/protein_domains.pq', engine='pyarrow')

In [ ]:
#| notest
# The same data, fetched live from Ensembl.
domain_df = targetdata.build_translation_overlap_df(aa_seq_df['id'].unique(), n_jobs=2)

In [ ]:
domain_feature_df = targetfeat.get_protein_domain_features(design_targ_df, domain_df, sources=None,
                                                           id_cols=id_cols)

In [ ]:
## conservation
conservation_df = pd.read_parquet('test_data/target_data/conservation.pq', engine='pyarrow')

In [ ]:
#| notest
# The same data, fetched live from UCSC.
conservation_df = targetdata.build_conservation_df(design_df, n_jobs=max_n_jobs)

In [ ]:
conservation_feature_df = targetfeat.get_conservation_features(design_targ_df, conservation_df,
                                                             small_width=2, large_width=16,
                                                             conservation_column='ranked_conservation',
                                                             id_cols=id_cols)
conservation_feature_df

In [ ]:
predictions = predict_target(design_df=design_df,
                             aa_subseq_df=aa_subseq_df,
                             domain_feature_df=domain_feature_df,
                             conservation_feature_df=conservation_feature_df)
design_df['Target Score'] = predictions

/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator SimpleImputer from version 1.0.dev0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator Pipeline from version 1.0.dev0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:443: UserWarning: X has feature names, but SimpleImputer was f

In [ ]:
lite_predictions = predict_target(design_df=design_df,
                                  aa_subseq_df=aa_subseq_df)
design_df['Target Score Lite'] = lite_predictions

/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator SimpleImputer from version 1.0.dev0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator Pipeline from version 1.0.dev0 when using version 1.0.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/modules/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/peterdeweirdt/miniforge3/envs/test_rs3_v6/lib/python3.9/site-packages/sklearn/base.py:443: UserWarning: X has feature names, but SimpleImputer was f

In [ ]:
assert stats.pearsonr(design_df['Target Score'], design_df['Target Score Lite'])[0] > 0.5

In [ ]:
sanger_df = pd.read_csv('test_data/Behan2019_activity.csv')
gecko_df = pd.read_csv('test_data/Aguirre2016_activity.csv')

sanger_designs = sanger_df.merge(design_df, how='inner',
                                 on=['sgRNA Sequence', 'sgRNA Context Sequence', 'Target Gene Symbol',
                                     'Target Cut %'])
gecko_designs = gecko_df.merge(design_df, how='inner',
                                on=['sgRNA Sequence', 'sgRNA Context Sequence', 'Target Gene Symbol',
                                    'Target Cut %'])
assert stats.pearsonr(sanger_designs['avg_mean_centered_neg_lfc'],
                      sanger_designs['Target Score'])[0] > 0.1